In [0]:
#Journey Mapping + Friction Point Analysis

In [0]:
#config + load
CATALOG = "cx"
SCHEMA_GOLD = "cx_gold"

from pyspark.sql.functions import (
    col, count, countDistinct, sum as sum_, avg, when,
    round as round_, datediff, date_trunc
)

events = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.journey_events")
segments = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.customer_segments")
customers = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.customer_360")

print(f"journey_events: {events.count():,}")
print(f"customer_segments: {segments.count():,}")

journey_events: 429,549
customer_segments: 50,000


In [0]:
#Journey stage summary
journey_stage_summary = (
    events
    .groupBy("journey_stage")
    .agg(
        count("*").alias("event_count"),
        countDistinct("customer_id").alias("unique_customers"),
    )
    .orderBy(col("event_count").desc())
)

journey_stage_summary.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.journey_stage_summary"
)
journey_stage_summary.show(truncate=False)

+-------------+-----------+----------------+
|journey_stage|event_count|unique_customers|
+-------------+-----------+----------------+
|Support      |262632     |47541           |
|Friction     |87172      |37316           |
|Onboarding   |50000      |50000           |
|Detractor    |10962      |8343            |
|Promoter     |9781       |7639            |
|Passive      |8846       |7033            |
|NULL         |156        |156             |
+-------------+-----------+----------------+



In [0]:
#Segment friction profile
events_with_seg = events.join(segments, "customer_id", "left")

friction_per_customer = (
    events_with_seg
    .filter(col("journey_stage") == "Friction")
    .groupBy("customer_id", "segment_name")
    .agg(count("*").alias("friction_count"))
)

segment_friction_profile = (
    friction_per_customer
    .groupBy("segment_name")
    .agg(
        count("*").alias("customers_with_friction"),
        round_(avg("friction_count"), 2).alias("avg_friction_per_customer"),
        sum_("friction_count").alias("total_friction_events"),
    )
    .orderBy(col("avg_friction_per_customer").desc())
)

segment_friction_profile.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.segment_friction_profile"
)
segment_friction_profile.show(truncate=False)

+--------------------+-----------------------+-------------------------+---------------------+
|segment_name        |customers_with_friction|avg_friction_per_customer|total_friction_events|
+--------------------+-----------------------+-------------------------+---------------------+
|At-Risk High-Touch  |8336                   |3.62                     |30181                |
|Dormant / Disengaged|7050                   |2.19                     |15407                |
|Strategic Champions |6175                   |2.0                      |12336                |
|Silent Majority     |15755                  |1.86                     |29248                |
+--------------------+-----------------------+-------------------------+---------------------+



In [0]:
#NPS funnel by tenure
survey_events = (
    events
    .filter(col("event_type") == "survey_response")
    .join(customers.select("customer_id", "contract_start_date"), "customer_id")
    .withColumn("tenure_days_at_survey", datediff(col("event_date"), col("contract_start_date")))
    .withColumn(
        "tenure_bucket",
        when(col("tenure_days_at_survey") <= 90, "0-3 months")
        .when(col("tenure_days_at_survey") <= 365, "3-12 months")
        .when(col("tenure_days_at_survey") <= 730, "1-2 years")
        .otherwise("2+ years")
    )
)

nps_by_tenure = (
    survey_events
    .filter(col("journey_stage").isin("Detractor", "Passive", "Promoter"))
    .groupBy("tenure_bucket", "journey_stage")
    .agg(count("*").alias("n"))
    .orderBy("tenure_bucket", "journey_stage")
)

nps_by_tenure.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.nps_by_tenure"
)
nps_by_tenure.show()


+-------------+-------------+----+
|tenure_bucket|journey_stage|   n|
+-------------+-------------+----+
|   0-3 months|    Detractor|   7|
|   0-3 months|      Passive|   7|
|   0-3 months|     Promoter|  10|
|    1-2 years|    Detractor|1370|
|    1-2 years|      Passive|1081|
|    1-2 years|     Promoter|1276|
|     2+ years|    Detractor|9149|
|     2+ years|      Passive|7445|
|     2+ years|     Promoter|8172|
|  3-12 months|    Detractor| 436|
|  3-12 months|      Passive| 313|
|  3-12 months|     Promoter| 323|
+-------------+-------------+----+



In [0]:
#Monthly friction trend
monthly_friction = (
    events
    .filter(col("journey_stage") == "Friction")
    .withColumn("event_month", date_trunc("month", col("event_date")))
    .groupBy("event_month")
    .agg(
        count("*").alias("friction_events"),
        countDistinct("customer_id").alias("affected_customers"),
    )
    .orderBy("event_month")
)

monthly_friction.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.monthly_friction_trend"
)
monthly_friction.show(50, truncate=False)

+-------------------+---------------+------------------+
|event_month        |friction_events|affected_customers|
+-------------------+---------------+------------------+
|2024-01-01 00:00:00|3205           |3085              |
|2024-02-01 00:00:00|2948           |2842              |
|2024-03-01 00:00:00|3288           |3148              |
|2024-04-01 00:00:00|3086           |2970              |
|2024-05-01 00:00:00|3090           |2966              |
|2024-06-01 00:00:00|3065           |2942              |
|2024-07-01 00:00:00|3184           |3049              |
|2024-08-01 00:00:00|3136           |3001              |
|2024-09-01 00:00:00|3138           |2999              |
|2024-10-01 00:00:00|3172           |3031              |
|2024-11-01 00:00:00|3018           |2899              |
|2024-12-01 00:00:00|3265           |3126              |
|2025-01-01 00:00:00|3211           |3084              |
|2025-02-01 00:00:00|2776           |2671              |
|2025-03-01 00:00:00|3129      

In [0]:
#Friction by issue category
tickets = spark.table(f"{CATALOG}.cx_silver.service_interactions")

friction_by_category = (
    tickets
    .filter(col("priority").isin("Critical", "High"))
    .filter(col("dq_flag_orphan") == 0)
    .groupBy("issue_category")
    .agg(
        count("*").alias("friction_tickets"),
        sum_("escalated_flag").alias("escalated_count"),
        round_(avg("resolution_hours"), 2).alias("avg_resolution_hrs"),
    )
    .orderBy(col("friction_tickets").desc())
)

friction_by_category.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.friction_by_category"
)
friction_by_category.show(truncate=False)

+----------------------+----------------+---------------+------------------+
|issue_category        |friction_tickets|escalated_count|avg_resolution_hrs|
+----------------------+----------------+---------------+------------------+
|Calibration           |11042           |1293           |49.51             |
|Instrument Malfunction|10983           |1350           |48.08             |
|Consumables Issue     |10956           |1357           |51.58             |
|Billing               |10937           |1385           |49.82             |
|Training Request      |10917           |1295           |49.95             |
|Connectivity          |10816           |1295           |49.86             |
|Software Bug          |10797           |1317           |50.04             |
|Other                 |10724           |1253           |50.79             |
+----------------------+----------------+---------------+------------------+



In [0]:
%sql
--Sanity check 1: stage distribution
SELECT journey_stage, event_count, unique_customers
FROM cx.cx_gold.journey_stage_summary
ORDER BY event_count DESC

journey_stage,event_count,unique_customers
Support,262632,47541
Friction,87172,37316
Onboarding,50000,50000
Detractor,10962,8343
Promoter,9781,7639
Passive,8846,7033
null,156,156


In [0]:
%sql
--Top finding: friction by segment
SELECT segment_name, customers_with_friction,
        avg_friction_per_customer, total_friction_events
 FROM cx.cx_gold.segment_friction_profile
 ORDER BY avg_friction_per_customer DESC

segment_name,customers_with_friction,avg_friction_per_customer,total_friction_events
At-Risk High-Touch,8336,3.62,30181
Dormant / Disengaged,7050,2.19,15407
Strategic Champions,6175,2.0,12336
Silent Majority,15755,1.86,29248


In [0]:
%sql 
--NPS by tenure bucket
SELECT tenure_bucket, journey_stage, n
FROM cx.cx_gold.nps_by_tenure
ORDER BY tenure_bucket, journey_stage

tenure_bucket,journey_stage,n
0-3 months,Detractor,7
0-3 months,Passive,7
0-3 months,Promoter,10
1-2 years,Detractor,1370
1-2 years,Passive,1081
1-2 years,Promoter,1276
2+ years,Detractor,9149
2+ years,Passive,7445
2+ years,Promoter,8172
3-12 months,Detractor,436


In [0]:
%sql
--Top friction categories
SELECT issue_category, friction_tickets, escalated_count, avg_resolution_hrs
FROM cx.cx_gold.friction_by_category
ORDER BY friction_tickets DESC


issue_category,friction_tickets,escalated_count,avg_resolution_hrs
Calibration,11042,1293,49.51
Instrument Malfunction,10983,1350,48.08
Consumables Issue,10956,1357,51.58
Billing,10937,1385,49.82
Training Request,10917,1295,49.95
Connectivity,10816,1295,49.86
Software Bug,10797,1317,50.04
Other,10724,1253,50.79
